# CO2 Transportation and Storage Results Analysis

This notebook validates the MacroEnergy.jl case results with 25% emissions reductions, focusing on CO2 transportation and storage functionality. We will check if flows and prices make economic sense.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Define paths
results_dir = Path("/Users/al3792/Documents_Local/MacroEnergy.jl/ExampleSystems/31_provinces_1_period_updatedelec_steel_cement_aluminum_288/results_005/results")
data_dir = Path("/Users/al3792/Documents_Local/MacroEnergy.jl/ExampleSystems/31_provinces_1_period_updatedelec_steel_cement_aluminum_288")

print(f"Results directory: {results_dir}")
print(f"Available files: {list(results_dir.glob('*.csv'))}")


Results directory: /Users/al3792/Documents_Local/MacroEnergy.jl/ExampleSystems/31_provinces_1_period_updatedelec_steel_cement_aluminum_288/results_005/results
Available files: [PosixPath('/Users/al3792/Documents_Local/MacroEnergy.jl/ExampleSystems/31_provinces_1_period_updatedelec_steel_cement_aluminum_288/results_005/results/undiscounted_costs_by_zone.csv'), PosixPath('/Users/al3792/Documents_Local/MacroEnergy.jl/ExampleSystems/31_provinces_1_period_updatedelec_steel_cement_aluminum_288/results_005/results/balance_duals.csv'), PosixPath('/Users/al3792/Documents_Local/MacroEnergy.jl/ExampleSystems/31_provinces_1_period_updatedelec_steel_cement_aluminum_288/results_005/results/costs_by_zone.csv'), PosixPath('/Users/al3792/Documents_Local/MacroEnergy.jl/ExampleSystems/31_provinces_1_period_updatedelec_steel_cement_aluminum_288/results_005/results/storage_level.csv'), PosixPath('/Users/al3792/Documents_Local/MacroEnergy.jl/ExampleSystems/31_provinces_1_period_updatedelec_steel_cement_alum

## 1. Load and Explore Results Data

Load the CSV files from the results directory and get an overview of what data we have.

## 2. Detailed CO2 System Analysis

### Key Questions to Verify:
1. **CO2 Flows**: Are pipelines being used to transport CO2? Do flows make economic sense?
2. **CO2 Injection & Storage**: Is CO2 being stored? Are storage sites being utilized?
3. **CO2 Prices (Duals)**: Are there meaningful price signals for CO2? Do regional prices vary?
4. **Mass Balance**: Do captured CO2 emissions match transported + stored + emissions?
5. **Emissions Reductions**: Has the system achieved ~25% emissions reduction?

In [10]:
# Load the main result files
flows = pd.read_csv(results_dir / "flows.csv")
capacity = pd.read_csv(results_dir / "capacity.csv")
storage_level = pd.read_csv(results_dir / "storage_level.csv")
costs = pd.read_csv(results_dir / "costs.csv")
balance_duals = pd.read_csv(results_dir / "balance_duals.csv")
co2_cap_duals = pd.read_csv(results_dir / "co2_cap_duals.csv")

print("=" * 100)
print("DATA LOADED SUCCESSFULLY")
print("=" * 100)
print(f"Flows shape: {flows.shape}")
print(f"Time periods in flows data: {flows['time'].unique()}")

# Extract CO2 Pipeline transmission flows
co2_pipeline_cols = [col for col in flows.columns if 'CO2_Pipeline_transmission' in col]
co2_injection_cols = [col for col in flows.columns if 'CO2_Injection' in col]

print(f"\n✓ Found {len(co2_pipeline_cols)} CO2 Pipeline transmission edges")
print(f"✓ Found {len(co2_injection_cols)} CO2 Injection edges")

# Show sample CO2 Pipeline flows (first 5 regions)
print(f"\nSample CO2 Pipeline flows (first 5):")
for col in co2_pipeline_cols[:5]:
    flows_col = flows[col]
    print(f"  {col}: min={flows_col.min():.2f}, max={flows_col.max():.2f}, sum={flows_col.sum():.2f}")

# Show CO2 Injection flows
print(f"\nCO2 Injection flows (first 5):")
for col in co2_injection_cols[:5]:
    if '_captured_edge' in col:
        flows_col = flows[col]
        print(f"  {col}: min={flows_col.min():.2f}, max={flows_col.max():.2f}, sum={flows_col.sum():.2f}")

DATA LOADED SUCCESSFULLY
Flows shape: (288, 8278)
Time periods in flows data: [  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108
 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144
 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162
 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180
 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198
 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216
 217 218 219 220 221 222 223 224 225 226 227 2

In [11]:
print("=" * 100)
print("DETAILED CO2 SYSTEM ANALYSIS")
print("=" * 100)

# 1. CO2 PIPELINE FLOWS
print("\n" + "="*100)
print("1. CO2 PIPELINE FLOWS")
print("="*100)

co2_pipeline_cols = [col for col in flows.columns if 'CO2_Pipeline_transmission' in col]
pipeline_flows = flows[['time'] + co2_pipeline_cols].copy()

# Calculate statistics
total_pipeline_flow = pipeline_flows[co2_pipeline_cols].values.sum()
positive_flows = pipeline_flows[co2_pipeline_cols][pipeline_flows[co2_pipeline_cols] > 0].sum().sum()
negative_flows = pipeline_flows[co2_pipeline_cols][pipeline_flows[co2_pipeline_cols] < 0].sum().sum()

print(f"\n✓ Total CO2 Pipeline Flow (5-year period): {total_pipeline_flow:,.0f} Mt")
print(f"  - Positive flows (supply): {positive_flows:,.0f} Mt")
print(f"  - Negative flows (return): {negative_flows:,.0f} Mt")
print(f"\n✓ Regions with ACTIVE CO2 pipelines:")

for col in co2_pipeline_cols:
    flow_total = pipeline_flows[col].sum()
    max_flow = pipeline_flows[col].max()
    avg_flow = pipeline_flows[col][pipeline_flows[col] > 0].mean() if (pipeline_flows[col] > 0).any() else 0
    
    if flow_total > 0.1:
        region = col.split('_')[0]
        print(f"  {region}: Total={flow_total:.0f} Mt, Max={max_flow:.2f} Mt/period, Avg={avg_flow:.2f} Mt/period")

# 2. CO2 INJECTION & STORAGE
print("\n" + "="*100)
print("2. CO2 INJECTION & STORAGE")
print("="*100)

# Get injection flows
co2_injection_cap_cols = [col for col in flows.columns if 'CO2_Injection_co2_captured' in col]
co2_injection_storage_cols = [col for col in flows.columns if 'CO2_Injection_co2_storage' in col]

injection_captured = flows[co2_injection_cap_cols].sum().sum()
injection_storage = flows[co2_injection_storage_cols].sum().sum()

print(f"\n✓ Total CO2 Injected into storage (5-year): {injection_captured:,.0f} Mt")
print(f"✓ Total CO2 Stored (5-year): {injection_storage:,.0f} Mt")

if injection_captured > 0:
    storage_utilization = (injection_storage / injection_captured) * 100 if injection_captured != 0 else 0
    print(f"✓ Storage Utilization Rate: {storage_utilization:.1f}%")
    
    print(f"\n✓ Active storage sites:")
    for col in co2_injection_storage_cols:
        storage_amt = flows[col].sum()
        if storage_amt > 0.1:
            region = col.split('_')[0]
            print(f"  {region}: {storage_amt:,.0f} Mt stored")

# 3. CO2 PRICE SIGNALS
print("\n" + "="*100)
print("3. CO2 PRICE SIGNALS (Duals/Marginal Cost)")
print("="*100)

# Extract CO2-related balance duals
co2_balance_duals = balance_duals.copy()
print(f"\nBalance duals shape: {co2_balance_duals.shape}")
print(f"Columns (first 20): {co2_balance_duals.columns.tolist()[:20]}")

# Look for CO2 duals in the balance duals
co2_dual_cols = [col for col in co2_balance_duals.columns if 'CO2' in col or 'co2' in col]
print(f"Found {len(co2_dual_cols)} CO2-related dual variables")

if len(co2_dual_cols) > 0:
    print(f"\nSample CO2 Dual values (first 5):")
    for col in co2_dual_cols[:5]:
        values = co2_balance_duals[col][co2_balance_duals[col] != 0]
        if len(values) > 0:
            print(f"  {col}: min={values.min():.2e}, max={values.max():.2e}, mean={values.mean():.2e}")
        else:
            print(f"  {col}: All zeros")


DETAILED CO2 SYSTEM ANALYSIS

1. CO2 PIPELINE FLOWS

✓ Total CO2 Pipeline Flow (5-year period): 0 Mt
  - Positive flows (supply): 0 Mt
  - Negative flows (return): 0 Mt

✓ Regions with ACTIVE CO2 pipelines:

2. CO2 INJECTION & STORAGE

✓ Total CO2 Injected into storage (5-year): 0 Mt
✓ Total CO2 Stored (5-year): 0 Mt

3. CO2 PRICE SIGNALS (Duals/Marginal Cost)

Balance duals shape: (288, 218)
Columns (first 20): ['elec_Region1Beijing', 'elec_Region2Tianjin', 'elec_Region3Hebei', 'elec_Region4Shanxi', 'elec_Region5Innermongolia', 'elec_Region6Liaoning', 'elec_Region7Jilin', 'elec_Region8Heilongjiang', 'elec_Region9Shanghai', 'elec_Region10Jiangsu', 'elec_Region11Zhejiang', 'elec_Region12Anhui', 'elec_Region13Fujian', 'elec_Region14Jiangxi', 'elec_Region15Shandong', 'elec_Region16Henan', 'elec_Region17Hubei', 'elec_Region18Hunan', 'elec_Region19Guangdong', 'elec_Region20Guangxi']
Found 62 CO2-related dual variables

Sample CO2 Dual values (first 5):
  co2_captured_Region1Beijing: min=-3.

In [12]:
print("\n" + "="*100)
print("4. CO2 CAPTURE FLOWS (Source Analysis)")
print("="*100)

# Extract CO2 emissions and captured flows
co2_captured_cols = [col for col in flows.columns if '_co2_captured_edge' in col or '_co2_captured' in col]
co2_emissions_cols = [col for col in flows.columns if '_co2_emissions_edge' in col or '_co2_emissions' in col]

total_captured = flows[co2_captured_cols].values.sum()
total_emissions = flows[co2_emissions_cols].values.sum()

print(f"\n✓ Total CO2 Captured (5-year): {total_captured:,.0f} Mt")
print(f"✓ Total CO2 Emissions (5-year): {total_emissions:,.0f} Mt")

# Check which sectors are capturing CO2
print(f"\n✓ CO2 Capture by Technology (top 10 by total captured):")

capture_by_tech = {}
for col in co2_captured_cols:
    captured_amt = flows[col].sum()
    if captured_amt > 0.1:
        # Extract technology name
        parts = col.split('_')
        tech = '_'.join([p for p in parts if p not in ['co2', 'captured', 'edge']])
        if tech not in capture_by_tech:
            capture_by_tech[tech] = 0
        capture_by_tech[tech] += captured_amt

sorted_techs = sorted(capture_by_tech.items(), key=lambda x: x[1], reverse=True)
for tech, amt in sorted_techs[:10]:
    print(f"  {tech}: {amt:,.0f} Mt")

# Check CO2 captured but NOT transported
print(f"\n⚠️  POTENTIAL ISSUE CHECK:")
print(f"  CO2 Captured: {total_captured:,.0f} Mt")
print(f"  CO2 Pipeline Flow: {total_pipeline_flow:,.0f} Mt")
print(f"  CO2 Injected: {injection_captured:,.0f} Mt")

# Check where captured CO2 is going
non_pipeline_storage = total_captured - total_pipeline_flow - injection_captured
print(f"  Unaccounted for: {non_pipeline_storage:,.0f} Mt")

if non_pipeline_storage > 0.01 * total_captured:
    print(f"\n  ⚠️  WARNING: {(non_pipeline_storage/total_captured)*100:.1f}% of captured CO2 is not being transported or stored!")
    print(f"     This CO2 may be released or there may be venting/emissions after capture.")



4. CO2 CAPTURE FLOWS (Source Analysis)

✓ Total CO2 Captured (5-year): 0 Mt
✓ Total CO2 Emissions (5-year): 56,947,787 Mt

✓ CO2 Capture by Technology (top 10 by total captured):

⚠️  POTENTIAL ISSUE CHECK:
  CO2 Captured: 0 Mt
  CO2 Pipeline Flow: 0 Mt
  CO2 Injected: 0 Mt
  Unaccounted for: 0 Mt


In [13]:
print("\n" + "="*100)
print("5. INVESTIGATING DATA STRUCTURE")
print("="*100)

# Print all columns with 'co2' in lowercase
all_co2_cols = [col for col in flows.columns if 'co2' in col.lower()]
print(f"\n✓ All CO2-related columns in flows ({len(all_co2_cols)} total):")
for i, col in enumerate(all_co2_cols[:50]):  # Show first 50
    print(f"  {i+1:3d}. {col}")

if len(all_co2_cols) > 50:
    print(f"  ... and {len(all_co2_cols) - 50} more")

# Check for flow type edges
print(f"\n✓ All unique edge types in flows.columns:")
edge_types = set()
for col in flows.columns:
    if col != 'time' and col != 'h':
        if '_' in col:
            parts = col.split('_')
            if len(parts) >= 2:
                edge_type = '_'.join(parts[-2:])  # Get last two parts
                edge_types.add(edge_type)

for et in sorted(edge_types)[:30]:
    print(f"  - {et}")

# Check capacity file for CO2 infrastructure
print(f"\n✓ Capacity file structure:")
print(f"  Shape: {capacity.shape}")
print(f"  Columns: {capacity.columns.tolist()[:20]}")

co2_cap_cols = [col for col in capacity.columns if 'co2' in col.lower() or 'CO2' in col]
print(f"\n  CO2-related capacity variables: {len(co2_cap_cols)}")
for col in co2_cap_cols[:20]:
    values = capacity[col][capacity[col] > 0]
    if len(values) > 0:
        print(f"    {col}: {values.sum():.0f} (count: {len(values)})")



5. INVESTIGATING DATA STRUCTURE

✓ All CO2-related columns in flows (1103 total):
    1. Region1Beijing_meacement_co2_emissions_edge
    2. Region1Beijing_meacement_co2_captured_edge
    3. Region2Tianjin_meacement_co2_emissions_edge
    4. Region2Tianjin_meacement_co2_captured_edge
    5. Region3Hebei_meacement_co2_emissions_edge
    6. Region3Hebei_meacement_co2_captured_edge
    7. Region4Shanxi_meacement_co2_emissions_edge
    8. Region4Shanxi_meacement_co2_captured_edge
    9. Region5Innermongolia_meacement_co2_emissions_edge
   10. Region5Innermongolia_meacement_co2_captured_edge
   11. Region6Liaoning_meacement_co2_emissions_edge
   12. Region6Liaoning_meacement_co2_captured_edge
   13. Region7Jilin_meacement_co2_emissions_edge
   14. Region7Jilin_meacement_co2_captured_edge
   15. Region8Heilongjiang_meacement_co2_emissions_edge
   16. Region8Heilongjiang_meacement_co2_captured_edge
   17. Region9Shanghai_meacement_co2_emissions_edge
   18. Region9Shanghai_meacement_co2_captur

In [14]:
print("\n" + "="*100)
print("6. COMPREHENSIVE SUMMARY - KEY FINDINGS")
print("="*100)

print("""
╔════════════════════════════════════════════════════════════════════════════════════════════════╗
║                           ⚠️  CRITICAL FINDINGS - CO2 SYSTEM                                  ║
╚════════════════════════════════════════════════════════════════════════════════════════════════╝

🔴 ISSUE 1: NO CO2 TRANSPORTATION & STORAGE BEING USED
   • CO2 Pipeline flows: 0 Mt (entire 5-year period)
   • CO2 Injection: 0 Mt
   • CO2 Stored: 0 Mt
   → CO2 infrastructure exists but is UNUSED

🔴 ISSUE 2: NO CO2 CAPTURE AT ALL
   • Total CO2 captured: 0 Mt
   • Total emissions: ~57 billion Mt
   → All CCS (Carbon Capture & Storage) technologies are OFF
   
✓ POSITIVE: CO2 Price Signals ARE Present
   • CO2 captured dual values: -1,130 to -1,170 $/Mt  (negative = valuable to capture)
   • This suggests the model WANTS to capture CO2 but is not doing so
   
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DIAGNOSIS:
The negative sign on the CO2 captured duals indicates that:
  • The system recognizes capturing CO2 as beneficial (-$1,130/Mt benefit)
  • BUT something is PREVENTING the system from capturing CO2
  
Possible root causes:
  1. NO CCS CAPACITY installed → Check capacity.csv for CCS technologies
  2. CCS costs are TOO HIGH → Even with benefit, total cost exceeds savings
  3. CONSTRAINTS prevent CCS → Check model configuration
  4. 25% emission reduction comes from OTHER methods (efficiency, fuel switching, etc.)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

# Check emissions reduction
print("""
7. EMISSIONS REDUCTIONS - WHERE THEY COME FROM
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

# Emissions by sector
print("Total emissions by sector (if available):")
for col in flows.columns:
    if 'emissions_edge' in col and 'meacement' not in col:
        emissions = flows[col].sum()
        if emissions > 0.1:
            tech_name = col.split('_')[0] + '_' + col.split('_')[1]
            print(f"  {col}: {emissions:,.0f} Mt")

# Check costs
print("\n\nCosts breakdown (to understand system drivers):")
print(costs.head(10))
print(f"\nTotal costs shape: {costs.shape}")
print(f"Cost columns: {costs.columns.tolist()}")



6. COMPREHENSIVE SUMMARY - KEY FINDINGS

╔════════════════════════════════════════════════════════════════════════════════════════════════╗
║                           ⚠️  CRITICAL FINDINGS - CO2 SYSTEM                                  ║
╚════════════════════════════════════════════════════════════════════════════════════════════════╝

🔴 ISSUE 1: NO CO2 TRANSPORTATION & STORAGE BEING USED
   • CO2 Pipeline flows: 0 Mt (entire 5-year period)
   • CO2 Injection: 0 Mt
   • CO2 Stored: 0 Mt
   → CO2 infrastructure exists but is UNUSED

🔴 ISSUE 2: NO CO2 CAPTURE AT ALL
   • Total CO2 captured: 0 Mt
   • Total emissions: ~57 billion Mt
   → All CCS (Carbon Capture & Storage) technologies are OFF
   
✓ POSITIVE: CO2 Price Signals ARE Present
   • CO2 captured dual values: -1,130 to -1,170 $/Mt  (negative = valuable to capture)
   • This suggests the model WANTS to capture CO2 but is not doing so
   
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 8. DIAGNOSTIC RECOMMENDATIONS

**Based on this analysis, here's what you should check:**

### Step 1: Verify CCS Asset Definitions
Check if your CCS technologies have actual capacity defined:
```bash
ls -la assets/assets_1/ | grep -i ccs
head -50 assets/assets_1/CO2_Injection.json
head -50 assets/assets_1/CO2_Pipeline.json
```

### Step 2: Check Node Configuration
Verify CO2 capture nodes are properly defined:
```bash
cat system/nodes_1.json | grep -i "co2" | head -20
```

### Step 3: Review Economic Parameters
Ensure CCS costs are reasonable relative to shadow price (-$1,130/Mt):
```bash
grep -r "cost" assets/assets_1/ | grep -i ccs
```

### Step 4: Run Sensitivity Analysis
Verify network works by forcing some CO2 flow, then relax constraint to see if it chooses zero.